In [1]:
!unzip "ลำปาง.zip"

Archive:  ลำปาง.zip
   creating: ลำปาง/
   creating: ลำปาง/นอกเขต/
   creating: ลำปาง/นอกเขต/ชุดที่ 1/
  inflating: ลำปาง/นอกเขต/ชุดที่ 1/ชุดที่ 1 ส.ส. 5-17(บช).pdf  
  inflating: ลำปาง/นอกเขต/ชุดที่ 1/ชุดที่ 1 ส.ส. 5-17.pdf  
   creating: ลำปาง/นอกเขต/ชุดที่ 10/
  inflating: ลำปาง/นอกเขต/ชุดที่ 10/ชุดที่ 10 5-17(บช).pdf  
  inflating: ลำปาง/นอกเขต/ชุดที่ 10/ชุดที่ 10 5-17.pdf  
   creating: ลำปาง/นอกเขต/ชุดที่ 11/
  inflating: ลำปาง/นอกเขต/ชุดที่ 11/ชุดที่ 11 5-17(บช).pdf  
  inflating: ลำปาง/นอกเขต/ชุดที่ 11/ชุดที่ 11 5-17.pdf  
   creating: ลำปาง/นอกเขต/ชุดที่ 12/
  inflating: ลำปาง/นอกเขต/ชุดที่ 12/ชุดที่ 12 5-17 (บช).pdf  
  inflating: ลำปาง/นอกเขต/ชุดที่ 12/ชุดที่ 12 5-17.pdf  
   creating: ลำปาง/นอกเขต/ชุดที่ 13/
  inflating: ลำปาง/นอกเขต/ชุดที่ 13/ชุดที่ 13 5-17 (บช).pdf  
  inflating: ลำปาง/นอกเขต/ชุดที่ 13/ชุดที่ 13 5-17.pdf  
   creating: ลำปาง/นอกเขต/ชุดที่ 2/
  inflating: ลำปาง/นอกเขต/ชุดที่ 2/ชุดที่ 2 ส.ส. 5-17 (บช).pdf  
  inflating: ลำปาง/นอกเขต/ชุดที่ 2/ชุดที่ 2 ส.ส. 5

In [2]:
!apt update
!apt install -y tesseract-ocr
!apt install -y tesseract-ocr-tha # Installs the Thai language data
!apt install -y libtesseract-dev
!pip install pytesseract PyMuPDF pillow

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:4 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,305 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,842 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,803 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,914 kB]
Get

In [14]:
import fitz
import pytesseract
from pathlib import Path
from PIL import Image
import os

# Define words to detect - we include both the spelling provided and the standard dictionary spelling just in case!
TARGET_WORDS = [
    "รายงาน",
    "บัตรดี",
    "บัตรเสีย",
    "พรรค",
    "ได้คะแนน",
    "หมายเลข"
]

source_dir = Path('ลำปาง')
output_dir = Path('page-ocr')

# Ensure output directory exists
output_dir.mkdir(parents=True, exist_ok=True)

# Find all PDF files
pdf_files = sorted(list(source_dir.glob('**/*.pdf')))
print(f"Found {len(pdf_files)} PDF files to process.")

for pdf_path in pdf_files:
    print(f"Processing: {pdf_path}")
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print(f"  Could not open {pdf_path}: {e}")
        continue

    found_pages = []

    # Iterate over all pages in the PDF
    for page_idx in range(len(doc)):
        page = doc.load_page(page_idx)

        # Convert PDF page to high-res image (300 DPI)
        zoom = 300 / 72.0
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # Extract text using Tesseract configured explicitly for Thai
        text = pytesseract.image_to_string(img, lang='tha')
        # print(text)
        # Check for target words
        word_found = False
        for target in TARGET_WORDS:
            # Remove any hidden trailing whitespace when checking inside the text
            if target.strip() in text:
                word_found = True
                break

        if word_found:
            found_pages.append((page_idx + 1, text))
            print(f"  - Target word detected on page {page_idx + 1}")

    doc.close()

    # Save results to txt file if any matches found
    if found_pages:
        output_filename = pdf_path.stem + ".txt"
        output_path = output_dir / output_filename
        # Handle filename collisions
        if output_path.exists():
            output_filename = f"{pdf_path.parent.name}_{pdf_path.stem}.txt"
            output_path = output_dir / output_filename

        with open(output_path, 'w', encoding='utf-8') as f:
            for page_num, extracted_text in found_pages:
                f.write(f"{page_num}\n")
                # f.write(extracted_text)
                # f.write("\n\n")
                # f.write(extracted_text)
                # f.write("\n\n")
        print(f"  Successfully saved matches to {output_path}")
    else:
        print(f"  No target word found in {pdf_path}")

print("Processing complete!")


Found 94 PDF files to process.
Processing: ลำปาง/นอกเขต/ชุดที่ 1/ชุดที่ 1 ส.ส. 5-17(บช).pdf
  - Target word detected on page 1
  - Target word detected on page 2
  - Target word detected on page 3
  Successfully saved matches to page-ocr/ชุดที่ 1 ส.ส. 5-17(บช).txt
Processing: ลำปาง/นอกเขต/ชุดที่ 1/ชุดที่ 1 ส.ส. 5-17.pdf
  - Target word detected on page 1
  Successfully saved matches to page-ocr/ชุดที่ 1 ส.ส. 5-17.txt
Processing: ลำปาง/นอกเขต/ชุดที่ 10/ชุดที่ 10 5-17(บช).pdf
  - Target word detected on page 1
  - Target word detected on page 3
  Successfully saved matches to page-ocr/ชุดที่ 10 5-17(บช).txt
Processing: ลำปาง/นอกเขต/ชุดที่ 10/ชุดที่ 10 5-17.pdf
  - Target word detected on page 1
  Successfully saved matches to page-ocr/ชุดที่ 10 5-17.txt
Processing: ลำปาง/นอกเขต/ชุดที่ 11/ชุดที่ 11 5-17(บช).pdf
  - Target word detected on page 1
  - Target word detected on page 2
  - Target word detected on page 3
  Successfully saved matches to page-ocr/ชุดที่ 11 5-17(บช).txt
Processing:

In [15]:
!zip -r page-ocr.zip page-ocr

  adding: page-ocr/ (stored 0%)
  adding: page-ocr/ชุดที่ 10 5-17.txt (stored 0%)
  adding: page-ocr/ชุดที่ 4 ส.ส. 5-17.txt (stored 0%)
  adding: page-ocr/สส.5-18 ตำบลบ้านอ้อน.txt (stored 0%)
  adding: page-ocr/แบบ ส.ส.5-18 (บช) ต.เมืองปาน.txt (deflated 42%)
  adding: page-ocr/ชุดที่ 4 ส.ส. 5-17(บช).txt (stored 0%)
  adding: page-ocr/แบบ ส.ส 5.18 ต.วิเชตนคร.txt (deflated 6%)
  adding: page-ocr/แบบ ส.ส 5.18 (บช) ต.วิเชตนคร.txt (deflated 43%)
  adding: page-ocr/สส.5-18 ตำบลหลวงเหนือ.txt (stored 0%)
  adding: page-ocr/แบบ ส.ส 5.18 ต.ทุ่งผึ้ง.txt (stored 0%)
  adding: page-ocr/สส.5-18บช ตำบลบ้านร้อง.txt (deflated 44%)
  adding: page-ocr/สส.5-18 ตำบลแม่ตีบ.txt (stored 0%)
  adding: page-ocr/ชุดที่ 2 ส.ส. 5-17 (บช).txt (stored 0%)
  adding: page-ocr/แบบ ส.ส 5.18 ต.บ้านสา.txt (stored 0%)
  adding: page-ocr/ชุดที่ 1 ส.ส. 5-17(บช).txt (stored 0%)
  adding: page-ocr/แบบ ส.ส.5-18 ต.เมืองปาน.txt (deflated 4%)
  adding: page-ocr/สส.5-18บช ตำบลบ้านแหง.txt (deflated 38%)
  adding: page-ocr/แบบ ส.ส. 5

In [16]:
!pip install -U -q google-colab-forced-cert PyDrive

ERROR: Could not find a version that satisfies the requirement google-colab-forced-cert (from versions: none)
ERROR: No matching distribution found for google-colab-forced-cert


In [17]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
# This will prompt you to authorize access to your Google Drive.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# Upload the zip file
file_name = 'page-ocr.zip'
gfile = drive.CreateFile({'title': file_name})
gfile.SetContentFile(file_name)
gfile.Upload()

print(f'Uploaded {file_name} to Google Drive.')

ModuleNotFoundError: No module named 'pydrive'